# 03 - Contrôle de conformité (verdict conforme / non-conforme)

**Objectif :** valider le module `src/quality/conformity.py`, qui compare un tracé
testé à la table de référence et rend un verdict, pièce par pièce et global.

**Tolérance retenue : ± 3 cm** sur chaque dimension (largeur, hauteur).

**Deux vérifications, pour prouver que le système fonctionne vraiment :**

1. **Test de cohérence** — comparer Tracee1 à sa propre référence. Comme c'est
   exactement les mêmes données, le résultat doit être 100% conforme, écart nul
   partout. Si ce n'est pas le cas, il y a un bug dans le pipeline.
2. **Test de détection de défaut** — on modifie artificiellement une pièce de
   Tracee1 (on l'agrandit de 5 cm, au-delà de la tolérance) pour simuler une
   pièce mal découpée, et on vérifie que le système la détecte correctement
   comme non conforme, sans se tromper sur les autres pièces.


In [1]:
import sys
sys.path.append('../src')

from quality.conformity import evaluate_tracee, print_report

REFERENCE_TABLE = '../data/processed/reference_dimensions.json'

## 1. Test de cohérence — Tracee1 comparé à sa propre référence

Résultat attendu : 100% conforme, 0 écart. C'est le test de base qui prouve que
le pipeline de mesure + comparaison ne contient pas d'erreur de calcul.

In [2]:
verdict_sain = evaluate_tracee(
    test_annotations_path='../data/processed/Tracee1_annotations.json',
    test_pdf_path='../data/raw/reference/Tracee1.pdf',
    reference_table_path=REFERENCE_TABLE,
    tolerance_cm=3.0,
)
print_report(verdict_sain)

assert verdict_sain.conforme_global, "ERREUR: le tracé de reference compare a lui-meme devrait etre 100% conforme !"
print("\nTest de coherence reussi : le pipeline ne fausse pas les mesures.")

RAPPORT DE CONFORMITE - Tracee1
Modele identifie : 157380CD-PDF
Tolerance appliquee : +/- 3.0 cm

OK  FACING-04 CHAD-RIGHT
OK  CHAD-WB
OK  M0YA44-UNDERFLY
OK  PATCH
OK  FACING-03 CHAD-LEFT PATCH
OK  CHAD-BACK
OK  TRUEREGULAR1-BACK PCKT
OK  M0YA44-
OK  PATCH PATCH
OK  M0YA44-LBACK10
OK  TRUEREGULAR1-BACK PCKT
OK  M0YA44-LBACK10
OK  YOKE CHAD-BACK
OK  CHAD-COIN PCKT-05
OK  PATCH
OK  PATCH
OK  PATCH
OK  PATCH
OK  PATCH
OK  PATCH
OK  PATCH
OK  PATCH
OK  M0YA44-FLY
OK  PASSANT
OK  M0YA44-LFRONT01

Taux de conformite : 100.0%
VERDICT GLOBAL : CONFORME

Test de coherence reussi : le pipeline ne fausse pas les mesures.


## 2. Test de détection de défaut

On crée une copie des annotations de Tracee1, mais on décale les points d'UNE
seule pièce pour l'agrandir de façon significative (bien au-delà des 3cm de
tolérance) — comme si cette pièce avait été mal découpée à l'atelier. On vérifie
que le système :
- détecte bien cette pièce comme non conforme
- ne se trompe pas sur les 24 autres pièces (toujours conformes)


In [3]:
import json
import copy

with open('../data/processed/Tracee1_annotations.json', encoding='utf-8') as f:
    data_defaut = json.load(f)

# on choisit une piece cible et on l'ETIRE artificiellement (pas juste translatee,
# sinon la largeur/hauteur ne changent pas : un decalage uniforme deplace la piece
# sans changer sa taille). On pousse uniquement les points a droite du centre vers
# la droite, ce qui augmente reellement la largeur mesuree.
PIECE_CIBLE = 'M0YA44-UNDERFLY'
DECALAGE_PX = 120  # ~6cm a l'echelle de Tracee1 (~0.0507 cm/px)

for piece in data_defaut['pieces']:
    if piece['nom'] == PIECE_CIBLE:
        xs = [x for x, y in piece['points_original_px']]
        centre_x = (min(xs) + max(xs)) / 2
        piece['points_original_px'] = [
            [x + DECALAGE_PX, y] if x > centre_x else [x, y]
            for x, y in piece['points_original_px']
        ]
        print(f"Piece '{PIECE_CIBLE}' etiree artificiellement de {DECALAGE_PX}px "
              f"(~{DECALAGE_PX*0.0507:.1f}cm de largeur en plus) pour simuler un defaut de decoupe.")

with open('../data/processed/Tracee1_annotations_AVEC_DEFAUT.json', 'w', encoding='utf-8') as f:
    json.dump(data_defaut, f, indent=2, ensure_ascii=False)

Piece 'M0YA44-UNDERFLY' etiree artificiellement de 120px (~6.1cm de largeur en plus) pour simuler un defaut de decoupe.


In [4]:
verdict_defaut = evaluate_tracee(
    test_annotations_path='../data/processed/Tracee1_annotations_AVEC_DEFAUT.json',
    test_pdf_path='../data/raw/reference/Tracee1.pdf',
    reference_table_path=REFERENCE_TABLE,
    tolerance_cm=3.0,
)
print_report(verdict_defaut)

RAPPORT DE CONFORMITE - Tracee1
Modele identifie : 157380CD-PDF
Tolerance appliquee : +/- 3.0 cm

OK  FACING-04 CHAD-RIGHT
OK  CHAD-WB
!!! M0YA44-UNDERFLY
       -> largeur: mesure 27.63cm vs reference 21.55cm (ecart 6.08cm > tolerance 3.0cm)
OK  PATCH
OK  FACING-03 CHAD-LEFT PATCH
OK  CHAD-BACK
OK  TRUEREGULAR1-BACK PCKT
OK  M0YA44-
OK  PATCH PATCH
OK  M0YA44-LBACK10
OK  TRUEREGULAR1-BACK PCKT
OK  M0YA44-LBACK10
OK  YOKE CHAD-BACK
OK  CHAD-COIN PCKT-05
OK  PATCH
OK  PATCH
OK  PATCH
OK  PATCH
OK  PATCH
OK  PATCH
OK  PATCH
OK  PATCH
OK  M0YA44-FLY
OK  PASSANT
OK  M0YA44-LFRONT01

Taux de conformite : 96.0%
VERDICT GLOBAL : NON CONFORME


In [5]:
# verification automatique du resultat attendu
piece_defaut = next(p for p in verdict_defaut.pieces if p.nom == PIECE_CIBLE)
autres_pieces = [p for p in verdict_defaut.pieces if p.nom != PIECE_CIBLE]

assert not piece_defaut.conforme, f"ERREUR: le defaut sur '{PIECE_CIBLE}' aurait du etre detecte !"
assert all(p.conforme for p in autres_pieces), "ERREUR: une piece saine a ete signalee a tort !"
assert not verdict_defaut.conforme_global, "ERREUR: le verdict global aurait du etre NON CONFORME !"

print(f"Test de detection reussi :")
print(f" - La piece modifiee ('{PIECE_CIBLE}') est correctement detectee NON CONFORME")
print(f" - Les {len(autres_pieces)} autres pieces restent correctement CONFORMES")
print(f" - Le verdict global est correctement NON CONFORME")

Test de detection reussi :
 - La piece modifiee ('M0YA44-UNDERFLY') est correctement detectee NON CONFORME
 - Les 24 autres pieces restent correctement CONFORMES
 - Le verdict global est correctement NON CONFORME


## Conclusion

Les deux tests passent : le pipeline mesure correctement les dimensions, compare
à la bonne référence (par modèle), applique la tolérance de ±3cm, et distingue
bien une pièce défectueuse des pièces saines — sans faux positif ni faux négatif
sur ce cas de test.

**Limite actuelle à garder en tête :** ce test utilise les points de contour
annotés à la main comme mesure. Le pipeline final devra utiliser les points
produits automatiquement par le modèle de détection (U-Net, Phase 3) à la place
de l'annotation manuelle — c'est cette brique de mesure automatique qui reste à
construire pour un vrai nouveau tracé de test.

## Prochaines étapes

1. Construire le pipeline de détection automatique des points de contour (U-Net)
2. Brancher ce pipeline en entrée de `evaluate_tracee()` à la place des annotations
   manuelles
3. Construire l'API FastAPI qui expose ce contrôle de conformité comme service
